In [ ]:
from mpi4py import MPI
# torch
import torch
import pickle
# quimb
import quimb.tensor as qtn
import autoray as ar

from vmc_torch.experiment.tn_model import fTNModel_reuse, fTNModel
from vmc_torch.experiment.tn_model import fTN_BFA_cluster_Model_reuse
from vmc_torch.hamiltonian_torch import spinful_Fermi_Hubbard_square_lattice_torch
from vmc_torch.torch_utils import SVD,QR_tao
import copy
# Register safe SVD and QR functions to torch
ar.register_function('torch','linalg.svd',SVD.apply)
ar.register_function('torch','linalg.qr',QR_tao.apply)
pwd = '/home/sijingdu/TNVMC/VMC_code/vmc_torch/data'

COMM = MPI.COMM_WORLD
SIZE = COMM.Get_size()
RANK = COMM.Get_rank()

# Hamiltonian parameters
Lx = int(8)
Ly = int(8)
# symmetry = 'U1_Z2'
symmetry = 'U1SU'
t = 1.0
U = 8.0
N_f = int(Lx*Ly-8)
n_fermions_per_spin = (N_f//2, N_f//2)
H = spinful_Fermi_Hubbard_square_lattice_torch(Lx, Ly, t, U, N_f, pbc=False, n_fermions_per_spin=n_fermions_per_spin)
graph = H.graph
# TN parameters
D = int(6)
chi = int(4*D)
dtype=torch.float64

def load_fpeps_vmc(skeleton_path, peps_params_path, scale=4.0):
    import pickle
    import quimb.tensor as qtn
    from symmray.fermionic_local_operators import FermionicOperator

    skeleton = pickle.load(open(skeleton_path, "rb"))
    peps_params = pickle.load(open(peps_params_path, "rb"))
    peps = qtn.unpack(peps_params, skeleton)
    # Precondition the fPEPS
    # 1. Sync the stored fermionic phases
    for ts in peps.tensors:
        ts.data.phase_sync(inplace=True) # phase info is stored in skeleton!
    # 2. Scale the tensor elements
    peps.apply_to_arrays(lambda x: torch.tensor(scale*x, dtype=dtype))
    # 3. Set the exponent to 0.0
    peps.exponent = 0.0
    # correct the format of oddpos
    for ts in peps.tensors:
        # for Z2 fPEPS converted from U1 fPEPS, need to correct the format of oddpos
        if ts.data.oddpos:
            oddpos = ts.data.oddpos
            nested_oddpos = oddpos[0].label[0]
            if isinstance(nested_oddpos, FermionicOperator):
                ts.data._oddpos = (nested_oddpos,)
    return peps

# Load PEPS
appendix = ''
if symmetry == 'U1_Z2':
    symmetry = 'Z2'
    appendix = '_U1'
elif symmetry == 'U1SU':
    symmetry = 'Z2'
    appendix = '_U1SU'

skeleton_path = pwd+f"/{Lx}x{Ly}/t={t}_U={U}/N={N_f}/{symmetry}/D={D}/peps_skeleton{appendix}.pkl"
peps_params_path = pwd+f"/{Lx}x{Ly}/t={t}_U={U}/N={N_f}/{symmetry}/D={D}/peps_su_params{appendix}.pkl"
peps = load_fpeps_vmc(skeleton_path, peps_params_path)
peps_params, peps_skeleton = qtn.pack(peps)
skeleton = pickle.load(open(pwd+f"/{Lx}x{Ly}/t={t}_U={U}/N={N_f}/{symmetry}/D={D}/peps_skeleton{appendix}.pkl", "rb"))
for ts in skeleton.tensors:
    ts.data._phases = {}

rpeps = copy.deepcopy(peps)

model_names = {
    fTNModel_reuse: f'fTN_reuse{appendix}',
}

# load benchmark model
model1 = fTNModel(
        peps,
        max_bond=-1,
)

init_step = int(1)
model2 = fTNModel_reuse(
        rpeps,
        max_bond=chi,
)
saved_model_params = torch.load(pwd+f'/{Lx}x{Ly}/t={t}_U={U}/N={N_f}/{symmetry}/D={D}/{model_names.get(type(model2), None)}/chi={chi}/model_params_step{init_step}.pth', weights_only=False)
saved_model_state_dict = saved_model_params['model_state_dict']
saved_model_params_vec = torch.tensor(saved_model_params['model_params_vec'])
model2.load_params(saved_model_params_vec)
rpeps_1 = model2.psi()
rpeps_1_params, rpeps_1_skeleton = qtn.pack(rpeps_1)
test_psi = qtn.unpack(rpeps_1_params, skeleton)

random_x = torch.tensor([2., 1., 2., 1., 2., 1., 2., 1., 1., 0., 1., 2., 1., 0., 2., 1., 2.,
       1., 2., 0., 2., 2., 2., 1., 1., 2., 1., 0., 1., 1., 0., 0., 2., 1.,
       2., 0., 2., 2., 1., 2., 2., 1., 1., 2., 2., 1., 2., 1., 1., 2., 1.,
       0., 1., 2., 1., 2., 1., 2., 2., 0., 3., 1., 2., 1.])
print(test_psi.get_amp(random_x).contract(), rpeps_1.get_amp(random_x).contract(), peps.get_amp(random_x).contract())


tensor(-4.8550e+10, dtype=torch.float64, grad_fn=<MulBackward0>) tensor(-4.8550e+10, dtype=torch.float64, grad_fn=<MulBackward0>) tensor(-3.1308e+10, dtype=torch.float64)


In [10]:
for i in range(len(peps.tensors)):
    print(peps.tensors[i].data.phases, rpeps.tensors[i].data.phases, model1.skeleton.tensors[i].data.phases, skeleton.tensors[i].data.phases)

{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
{} {} {} {}
